# 5. Troubleshooting — common failure modes

Five issues that come up the first few times you use SCRAPs, with
diagnoses you can apply directly. Skim by issue: the headings list
the symptom verbatim.

This notebook is reference material, not a sequential tutorial — open
to the section matching what you're hitting.


## "My fitness plateaus above zero and I can't see why"

Most often: the cell + composition combination makes the SQS target
mathematically infeasible. SCRAPs' fitness has a strictly positive
lower bound when `T_ij^k = N_i · z_k · N_j / N_τ` is non-integer for
any weighted shell-pair. The optimizer plateaus at √(Σ residuals²)
and stays there forever.

**Diagnose**: SCRAPs prints a warning at startup:

```
WARNING: 4 weighted target pair count(s) are non-integer
  for this (ELEMENT, Cell_DIM, MAXSHELLNUM).
  Minimum achievable fitness >= sqrt(...) = 1.58.
  Exact SQS (fitness = 0) is mathematically unreachable on this cell.
```

If you see this, the fitness floor IS what you're measuring — not an
optimizer failure. Two recovery paths:


In [1]:
# Path 1: pick a feasible cell automatically via suggest_cell.py
import subprocess, sys
from pathlib import Path
REPO = Path.cwd().parent
suggest = REPO / "tools" / "suggest_cell.py"

# Example: NiCoCr 25:50:25 — infeasible at 3x3x3 = 108, feasible at 4x4x4
fcc_poscar = '''FCC Cu
1.0
3.56 0.0 0.0
0.0 3.56 0.0
0.0 0.0 3.56
4
Direct
0.0 0.0 0.0
0.5 0.5 0.0
0.5 0.0 0.5
0.0 0.5 0.5
'''
(Path.cwd() / "POSCAR_fcc").write_text(fcc_poscar)
result = subprocess.run(
    [sys.executable, str(suggest), str(Path.cwd() / "POSCAR_fcc"),
     "--composition", "0.25,0.5,0.25", "--target-atoms", "120",
     "--max-shellnum", "2"],
    capture_output=True, text=True, timeout=30,
)
print(result.stdout)


**Path 2** (less common): accept the floor as physical, and report the
achieved structure as the closest-integer-feasible approximant.
Useful when you need EXACTLY this cell size for downstream DFT and
you're targeting α=0 but the integer constraint gives a small offset.

---

## "α(Ni-Cr) plateaus far from my literature target"

If your target is non-zero α at one shell, the optimizer hits exactly
that target ONLY when `WEIGHT` zeroes the non-targeted shells.
Otherwise it finds the compromise minimum across all weighted shells.

The reason is physical: perfect α=-1 at 2NN inherently forces α=±1 at
3NN, 5NN, etc. (3D bipartite ordering induces correlations at every
distance). If you weight all shells equally, the optimizer settles
for "α=-0.5 at 2NN, α=±0.5 at other shells" as the joint minimum.

**Fix**: weight only the targeted shell.


In [2]:
sys.path.insert(0, str(REPO / "tools"))
from scraps_ase import run_sqs_from_atoms
from ase.build import bulk

primitive = bulk("Cu", "fcc", a=3.56, cubic=True)
# Multi-shell weighting (target at NN only) makes alpha(Ni-Cr)=-1 unreachable:
# the optimizer balances NN target against 2NN/3NN incidental shells.
# WEIGHT [1.0, 0.0] tells SCRAPS "ignore shells 1+":
csro = run_sqs_from_atoms(
    primitive,
    species=["Ni", "Co", "Cr"],
    element_counts=[36, 36, 36],
    cell_dim=(3, 3, 3),
    max_shellnum=2,
    weights=[1.0, 0.0],   # <-- focus only on NN; ignore 2NN
    sro_targets=[(0, 2, 0, -0.4)],   # alpha(Ni-Cr) at NN = -0.4 (moderate)
    auto_budget=2,
    rng_seed=42,
    work_dir=Path.cwd() / "nb5_focused_weight",
)
print(f"Final fitness: {csro.info['scraps_final_fitness']:.4f}")


Final fitness: 0.5657


For very strong CSRO targets (|α| near 1), small cells lack the
integer pair-count resolution to bracket the target — you may need
3×3×3+ for fine targets. See the integer-feasibility footnote in
notebook 4 (the 2×2×2 fluorite case plateaued at α=+0.33 instead of
+0.5 for this reason; 3×3×3 converged exactly).

---

## "Multi-sublattice run aborts with 'non-uniform coordination'"

Two flavors:

1. **You declared one big SUBLATTICE containing mixed Wyckoff orbits.**
   Each SUBLATTICE block must contain only atoms with the same per-
   shell coordination — distinct orbits must be separate blocks.
   Run `tools/sublattice_detect.py POSCAR` to see the orbit
   partition; copy that to your INPUT.

2. **You used legacy single-sublattice mode on a non-uniform cell.**
   Fluorite, sigma, Laves, MAX phases all need SUBLATTICE blocks
   (one per Wyckoff orbit) for the per-Wyckoff path to handle the
   different per-orbit coordination numbers.

In both cases the diagnostic names the offending atom pair and
points at `tools/sublattice_detect.py`.

---

## "SCRAPS and verify_sro.py give different α numbers"

For multi-sublattice cases this is **expected** when you call
`verify_sro.py` without telling it about the sublattice partition.

`verify_sro.py` computes α against the **global** c_j = N_j/N. SCRAPs
internally uses the per-sublattice c_j^τ = N_j/N_τ (v0.4.1+).
For B2 with 50/50 per sublattice these differ by a factor of 2 (global
c=0.25, per-sub c=0.5), so `verify_sro` reports cross-sublattice α
values around ±1 trivially while SCRAPs sees them at 0.

**Fix**: use the per-sublattice α calculator from the comparison
harness, as notebook 4 does:

```python
from compare_b2_multisub import compute_alpha_per_shell
shells = compute_alpha_per_shell(
    poscar_path, species=[...], sublattice_of={...}, n_shells=3,
)
```

`verify_sro.py --spectator-species X` handles the HEB-spectator
case correctly (one designed sublattice + one spectator); the
general-multi-sublattice calculator is for designs with ≥ 2
disordered sublattices.

---

## "mcsqs and SCRAPS disagree on my CSRO target"

First check that both calculations use the same lattice, composition,
cluster definitions, occupational basis, targets, weights and computational
budget. Warren–Cowley SRO and decorated correlations are distinct observables.
Use `m_scraps compare-atat` for the supplied independent references, and
`m_scraps analyze` for explicit figure/target comparisons. A discrepancy can
indicate a convention mismatch, incomplete convergence, or a bug; it should
be investigated rather than explained by an assumed method-specific floor.
